# 1. 逻辑回归的多分类实现

In [1]:
from sklearn.datasets import load_iris
from torch.utils.data import Dataset, DataLoader   # 数据集的统一处理与分批次处理
from torch import nn     # 面向对象的封装
from torch import optim  # 梯度更新的封装
import torch

## 数据集

In [2]:
class IrisDataset(Dataset):
    """
    类的文档注释
    """
    def __init__(self):
        #调用父类构造器
        super(IrisDataset, self).__init__()   # python ABC
        # 加载数据
        data, target = load_iris(return_X_y=True)
        # 转换为张量
        self.x = torch.tensor(data[:], dtype=torch.float32)
        self.y = torch.tensor(target[:], dtype=torch.long).view(-1)

    def __getitem__(self, idx):
        """
        重载[]运算符
        参数：
            idx：访问数据的下标
        返回：
            指定位置的数据，样本与标签
        """
        return self.x[idx], self.y[idx]

    def __len__(self):
        return self.x.shape[0]   # len(self.x)

In [5]:
datas, targets = load_iris(return_X_y=True)
test = torch.tensor(targets[:], dtype=torch.long).view(-1)
test.shape

torch.Size([150])

## 模型

In [3]:
class MultClassifyModule(nn.Module):

    def __init__(self, in_f, out_f):
        super().__init__()
        self._fc = nn.Linear(in_f, out_f)
        self._activate = nn.Softmax(dim=1)

    def forward(self, x):
        p = self._fc(x)
        p = self._activate(p)
        return p

In [4]:
@torch.no_grad
def evaluate(model, loader):
    model.to(device)
    num_sample = 0
    num_current = 0.0
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        y_ = model(x)

        # 判定，统计每分类下的最大概率的下标
        cls_ids = y_.argmax(dim=1)
        num_current += (y == cls_ids),float().sum()
        num_sample += len(x)
    return num_current / num_sample

In [ ]:
def train(model, loader, loss_f, optimizer, epoch=10000):
    model.to(device)
    from tqdm.notebook import tqdm
    td = tqdm(range(epoch), desc="train..", colour="blue")
    for e in td:
        for x, y in loader:
            optimizer.zero_grad()
            x = x.to(device)
            y = y.to(device)
            y_ = model(x)
            loss = loss_f(y_, y) # 交叉熵怎么处理真实标签与向量类型的不匹配
            loss.backward()
            optimizer.step()
    if e % 1000 == 0:
        current_rate = evaluate(model, loader)
        print(F"轮数{e:05d}, 损失值:{loss.cpu().item():.6f}，准确率:{currect_rate.cpu().item()* 100: .6f}%")
        # td.postset({
        #     "E": f"{e:05d}",
        #     "loss": f"{loss.item():.6f}",
        #     "cur": f"{current_rate.item() * 100:.6f}"
        # })

## 训练

In [7]:
epoch = 10000
lr = 0.001

ds = IrisDataset()
loader = DataLoader(ds, batch_size=30, shuffle=True)

model = MultClassifyModule(4, 3)

loss_f = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    params=model.parameters(),  # 取出模型所有的参数， 也是优化器更新的目标
    lr=lr
)

device = "cuda:0" if torch.cuda.is_available() else "cpu"
model.to(device) # 移动模型到设备


MultClassifyModule(
  (_fc): Linear(in_features=4, out_features=3, bias=True)
  (_activate): Softmax(dim=1)
)

In [ ]:
train(model, loader, loss_f, optimizer)

train..:   0%|          | 0/10000 [00:00<?, ?it/s]